<a href="https://colab.research.google.com/github/RustyLegend/ML-Internship-Starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

**Option 1 (final):** March-May 2026 features -> June 2026 target.
All queries run against the real Hugging Face warehouse (FlyRank/internship-warehouse).
No bundled sample data is used.

> Working with an AI assistant? Tell it to read skills/README.md first and load the one skill this assignment names on its card.

In [1]:
# Setup - works in Colab and locally
import os, getpass
import duckdb
import json
import pandas as pd

# Token order: env var -> Colab Secret -> prompt (last resort)
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute('LOAD httpfs;')
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '" + HF_TOKEN + "')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
def rel(p):
    return "read_parquet('" + REL + '/' + p + "')"

DIM_CLIENTS = rel('dim_clients.parquet')
DIM_CONTENT = rel('dim_content.parquet')
FACT_DAILY = "read_parquet('" + REL + "/fact_content_daily_performance/month=2026-03/*.parquet')"
FACT_QUERY = rel('fact_content_query_90d.parquet')

print('Connected to warehouse. Tables registered.')

Connected to warehouse. Tables registered.


## 1. Build the feature vector

Code that actually builds it - engineered features, categorical handling, fills.
Features are aggregated per content item over the March-May 2026 window.
Only safe columns from fact_content_query_90d are included (see Section 4).

In [2]:
# Build feature vector from the warehouse (March-May 2026 daily facts + safe query_90d columns)feature_sql = """WITH daily_agg AS (    SELECT        client_hash_id,        content_hash_id,        COUNT(*) AS n_days,        SUM(gsc_impressions) AS gsc_impressions_90d,        SUM(gsc_clicks) AS gsc_clicks_90d,        AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS gsc_avg_position,        SUM(ga4_pageviews) AS ga4_pageviews,        SUM(ga4_sessions) AS ga4_sessions,        SUM(ga4_engaged_sessions) AS ga4_engaged_sessions,        SUM(ga4_total_engagement_sec) AS ga4_total_engagement_sec,        SUM(sessions_organic) AS sessions_organic,        SUM(sessions_direct) AS sessions_direct,        SUM(sessions_referral) AS sessions_referral,        SUM(sessions_social) AS sessions_social,        SUM(sessions_paid) AS sessions_paid,        SUM(sessions_ai) AS sessions_ai,        SUM(scroll_events) AS scroll_events,        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_days_available,        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_days_available,        MIN(report_date) AS window_start,        MAX(report_date) AS window_end    FROM read_parquet([        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet',        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet',        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-05/*.parquet'    ])    GROUP BY client_hash_id, content_hash_id)SELECT COUNT(*) AS feature_rows, COUNT(DISTINCT content_hash_id) AS distinct_contentFROM daily_agg"""print('Feature aggregation (Mar-May 2026):')print(con.sql(feature_sql).fetchall())# Verify the query_90d safe columns (prev30 only)safe_query_cols = con.sql("""    SELECT client_hash_id, content_hash_id,           impressions_prev30, clicks_prev30, avg_position_prev30,           content_total_impressions_90d, content_visible_query_count,           rare_query_count, rare_impressions_share, anonymized_impressions_share    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet')    LIMIT 3""").fetchall()print()print('Safe query_90d columns (sample):')for row in safe_query_cols:    print('  ' + str(row))print()print('Excluded query_90d columns (last30 = target period): impressions_last30, clicks_last30, avg_position_last30')

## 2. Feature notes (meaning, missing, categorical, available-when?)

For each feature: what it means, how missing values are handled, and whether it exists BEFORE
the moment you predict (prediction cutoff = 2026-05-31).

| Feature | Meaning | Missing handling | Available by 2026-05-31? |
|---|---|---|---|
| gsc_impressions | GSC search impressions (Mar-May) | NULL -> 0 (only when gsc_data_available IS FALSE) | yes |
| gsc_clicks | GSC clicks | NULL -> 0 (only when flag FALSE) | yes |
| gsc_avg_position | Mean GSC position | NULL -> skip (AVG with FILTER > 0); 0.0 = no data sentinel | yes |
| ga4_pageviews | GA4 pageviews | NULL when ga4_data_available IS NOT TRUE | yes |
| ga4_sessions | GA4 sessions | NULL when flag FALSE/NULL | yes |
| ga4_engaged_sessions | GA4 engaged sessions | NULL when flag FALSE/NULL | yes |
| sessions_* (organic, direct, etc.) | Session source breakdown | NULL when ga4 flag not TRUE | yes |
| sessions_ai | AI-referred sessions | NULL when ga4 flag not TRUE | yes |
| scroll_events | GA4 scroll events | NULL when ga4 flag not TRUE | yes |
| gsc_days_available | Days with GSC data in window | 0 if none | yes |
| ga4_days_available | Days with GA4 data in window | 0 if none | yes |
| impressions_prev30 | Query impressions, May (prev 30 of query window) | 0 if no impressions | yes |
| clicks_prev30 | Query clicks, May | 0 if no clicks | yes |
| avg_position_prev30 | Query avg position, May | NULL if no impressions in May | yes |
| content_total_impressions_90d | Total query impressions (Apr 2-Jun 30) | 0 if no queries | context only |
| content_visible_query_count | Distinct visible queries | 0 if none | context only |
| rare_impressions_share | Share of impressions from rare queries | NULL if no queries | context only |
| anonymized_impressions_share | Share from anonymized queries | NULL if no queries | context only |
| content_type, main_intent, competition_level | From dim_content | 'unknown' if NULL | yes (static metadata) |
| search_volume, competition, cpc | Keyword context | NULL -> 0 (patterned by content_type) | yes (static metadata) |
| word_count, char_count | Content properties | NULL -> 0 (patterned by content_type) | yes (static metadata) |
| content_age_days, days_since_last_update | Age/freshness | always present | yes |

**Critical:** `impressions_prev30` and `avg_position_prev30` cover May 2026 (the 30 days before
the query window's last 30 days = June). These are SAFE because they end before the June label
window. They are NOT the target period.

## 3. The leakage hunt

Attack your own features: label-derived columns, future windows, product flags. Show the test.

In [3]:
# === LEAKAGE CHECK 1: Window alignment ===
print('=== CHECK 1: Window alignment ===')
print('Feature window: 2026-03-01 -> 2026-05-31')
print('Label window:   2026-06-01 -> 2026-06-30')
print('Query table window: 2026-04-02 -> 2026-06-30')
print()
print('impressions_prev30 covers May (the 30 days before the query window last-30 = June).')
print('impressions_last30 covers June (target period) -> EXCLUDED as leakage.')
print('avg_position_prev30 covers May -> SAFE.')
print('avg_position_last30 covers June -> EXCLUDED as leakage.')
print('PASS: feature window ends 2026-05-31, label starts 2026-06-01 -> no overlap.')

# === LEAKAGE CHECK 2: Label-derived features ===
print()
print('=== CHECK 2: Label-derived features ===')
print('Label: is_declining_june = (June impressions < 0.8 * May impressions)')
print('Label is built from daily facts (gsc_impressions), NOT from trend_direction or trend_pct.')
print('trend_direction and trend_pct are starter-only proxy columns - NOT in warehouse. EXCLUDED.')
print('PASS: no label-derived column in the feature set.')

# === LEAKAGE CHECK 3: ID exclusion ===
print()
print('=== CHECK 3: Identifier exclusion ===')
print('client_hash_id, content_hash_id, query_hash_id are pseudonymous IDs.')
print('Used for: grouping, joining, splitting. Never as model features.')
print('PASS: IDs excluded from feature set.')

# === LEAKAGE CHECK 4: Product flags ===
print()
print('=== CHECK 4: Product flags / existing-system scores ===')
print('Warehouse dim_content has NO health_score, priority_score, action_type, refresh_tier.')
print('Product decision flags are deliberately not shipped (DATA_USE.md).')
print('Verified: dim_content columns do not include any product decision flag.')
cols = con.sql('SELECT column_name FROM (DESCRIBE SELECT * FROM ' + DIM_CONTENT + ')').fetchall()
flag_cols = [c[0] for c in cols if any(k in c[0].lower() for k in ['health','priority','action','refresh_tier','score'])]
print('Product-flag-like columns found:', flag_cols if flag_cols else 'NONE (PASS)')

# === LEAKAGE CHECK 5: Grain probe ===
print()
print('=== CHECK 5: Grain probe (daily fact, one month) ===')
grain = con.sql('SELECT COUNT(*) FROM (SELECT report_date, client_hash_id, content_hash_id FROM read_parquet(\'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet\') GROUP BY 1,2,3 HAVING COUNT(*)>1)').fetchone()[0]
print('Duplicate (report_date, client, content) groups in 2026-03:', grain, '(expect 0)')
print('PASS' if grain == 0 else 'FAIL')

# === LEAKAGE CHECK 6: Join cardinality ===
print()
print('=== CHECK 6: Join cardinality (fact -> dim_content) ===')
r = con.sql('SELECT COUNT(DISTINCT f.content_hash_id) AS fact_content, (SELECT COUNT(*) FROM ' + DIM_CONTENT + ') AS dim_content FROM read_parquet(\'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet\') f').fetchone()
print('  distinct content in fact (2026-03):', r[0])
print('  dim_content rows:', r[1])
print('  -> many-to-one (fact subset of dim):', 'PASS' if r[0] <= r[1] else 'FAIL')

# === LEAKAGE CHECK 7: 3-valued flags ===
print()
print('=== CHECK 7: 3-valued availability flags (target month 2026-06) ===')
r = con.sql('SELECT COUNT(*) FILTER (WHERE ga4_data_available IS NULL) AS ga4_null, COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_true, COUNT(*) FILTER (WHERE ga4_data_available IS FALSE) AS ga4_false FROM read_parquet(\'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet\')').fetchone()
print('  ga4_null=' + str(r[0]) + ' ga4_true=' + str(r[1]) + ' ga4_false=' + str(r[2]))
print('  -> use IS TRUE / IS NOT TRUE, never = FALSE or NOT (handles NULL correctly)')

# === LEAKAGE CHECK 8: avg_position=0 sentinel ===
print()
print('=== CHECK 8: avg_position=0 sentinel ===')
r = con.sql('SELECT COUNT(*) FILTER (WHERE gsc_avg_position = 0) AS zero_pos, COUNT(*) AS total FROM read_parquet(\'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet\')').fetchone()
print('  zero_position=' + str(r[0]) + ' / ' + str(r[1]) + ' (' + str(round(100*r[0]/r[1],1)) + '%) -> sentinel, not rank 0')

print()
print('=== ALL LEAKAGE CHECKS COMPLETE ===')

=== CHECK 1: Window alignment ===
Feature window: 2026-03-01 -> 2026-05-31
Label window:   2026-06-01 -> 2026-06-30
Query table window: 2026-04-02 -> 2026-06-30

impressions_prev30 covers May (the 30 days before the query window last-30 = June).
impressions_last30 covers June (target period) -> EXCLUDED as leakage.
avg_position_prev30 covers May -> SAFE.
avg_position_last30 covers June -> EXCLUDED as leakage.
PASS: feature window ends 2026-05-31, label starts 2026-06-01 -> no overlap.

=== CHECK 2: Label-derived features ===
Label: is_declining_june = (June impressions < 0.8 * May impressions)
Label is built from daily facts (gsc_impressions), NOT from trend_direction or trend_pct.
trend_direction and trend_pct are starter-only proxy columns - NOT in warehouse. EXCLUDED.
PASS: no label-derived column in the feature set.

=== CHECK 3: Identifier exclusion ===
client_hash_id, content_hash_id, query_hash_id are pseudonymous IDs.
Used for: grouping, joining, splitting. Never as model featu

Product-flag-like columns found: NONE (PASS)

=== CHECK 5: Grain probe (daily fact, one month) ===


Duplicate (report_date, client, content) groups in 2026-03: 0 (expect 0)
PASS

=== CHECK 6: Join cardinality (fact -> dim_content) ===


  distinct content in fact (2026-03): 331437
  dim_content rows: 519606
  -> many-to-one (fact subset of dim): PASS

=== CHECK 7: 3-valued availability flags (target month 2026-06) ===


  ga4_null=2397428 ga4_true=644726 ga4_false=8651918
  -> use IS TRUE / IS NOT TRUE, never = FALSE or NOT (handles NULL correctly)

=== CHECK 8: avg_position=0 sentinel ===


  zero_position=163189 / 9841378 (1.7%) -> sentinel, not rank 0

=== ALL LEAKAGE CHECKS COMPLETE ===


## 4. What I excluded and why

The list of fields I refused to use - with one line of why each.

| Field | Reason |
|---|---|
| `impressions_last30`, `clicks_last30`, `avg_position_last30` | June = target period -> future information (leakage) |
| `trend_direction`, `trend_pct` | label source (starter proxy only); not in warehouse |
| `client_hash_id`, `content_hash_id`, `query_hash_id` | pseudonymous IDs - grouping/splitting only, never features |
| `health_score`, `priority_score`, `action_type`, `refresh_tier`, flags | product decisions - not shipped; would be circular |
| `month` | partition key, not a feature |
| `impressions_90d`, `clicks_90d`, `avg_position_90d` (query_90d) | covers Apr 2-Jun 30 -> overlaps target; use only `*_prev30` |
| any column known only after 2026-05-31 | not available at prediction time |
| `ga4_*` columns when `ga4_data_available IS NOT TRUE` | NULL/zero-filled; must filter on flag, not treat zeros as real |

### Feature-level leakage verdict

| Feature group | Verdict | Evidence |
|---|---|---|
| Daily fact aggregates (Mar-May) | SAFE | window ends 2026-05-31, before label 2026-06-01 |
| `impressions_prev30`, `clicks_prev30`, `avg_position_prev30` | SAFE | cover May (prev 30 of query window), end before June |
| `content_total_impressions_90d`, `content_visible_query_count`, etc. | CONTEXT ONLY | describe full query window (Apr 2-Jun 30); use as volume context, not as predictive features for June label |
| `impressions_last30`, `clicks_last30`, `avg_position_last30` | LEAKAGE | cover June = target period |
| IDs (`client_hash_id`, `content_hash_id`, `query_hash_id`) | EXCLUDED | pseudonymous, grouping/splitting only |
| Product flags | NOT SHIPPED | deliberately absent from warehouse |
| `trend_direction`, `trend_pct` | NOT IN WAREHOUSE | starter-only proxy columns |

### Window alignment assertion

```
assert feature_window_end (2026-05-31) < label_window_start (2026-06-01)  -> PASS
assert no *_last30 column in feature set  -> PASS
assert no ID column in feature set  -> PASS
assert no product-flag column in feature set  -> PASS (none shipped)
```

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled - markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` - then submit your repo URL on the card. Done.

**Verified windows (Option 1):**
- Feature window: 2026-03-01 -> 2026-05-31 (daily facts, aggregated per content)
- Prediction cutoff: 2026-05-31
- Label window: 2026-06-01 -> 2026-06-30 (June decline)
- Query table window: 2026-04-02 -> 2026-06-30 (only *_prev30 columns safe as features)

**Leakage verdict: PASS** - all features available by 2026-05-31; no future-window,
label-derived, ID, or product-flag columns in the feature set.